In [1]:
%pip install torch kiwipiepy pandas scikit-learn

     ---------------------------------------- 0.0/87.9 MB ? eta -:--:--
      --------------------------------------- 2.1/87.9 MB 11.8 MB/s eta 0:00:08
     -- ------------------------------------- 4.7/87.9 MB 11.7 MB/s eta 0:00:08
     --- ------------------------------------ 7.1/87.9 MB 11.6 MB/s eta 0:00:07
     ---- ----------------------------------- 9.4/87.9 MB 11.7 MB/s eta 0:00:07
     ----- --------------------------------- 12.1/87.9 MB 11.7 MB/s eta 0:00:07
     ------ -------------------------------- 14.4/87.9 MB 11.7 MB/s eta 0:00:07
     ------- ------------------------------- 17.0/87.9 MB 11.7 MB/s eta 0:00:07
     -------- ------------------------------ 19.4/87.9 MB 11.7 MB/s eta 0:00:06
     --------- ----------------------------- 22.0/87.9 MB 11.7 MB/s eta 0:00:06
     ---------- ---------------------------- 24.4/87.9 MB 11.7 MB/s eta 0:00:06
     ----------- --------------------------- 26.7/87.9 MB 11.7 MB/s eta 0:00:06
     ------------- ------------------------- 29.

In [2]:
"""
영화 리뷰 긍정/부정 분류: 공개 데이터(NSMC) + 형태소(Kiwi) + LSTM (PyTorch)

데이터: 네이버 영화 리뷰 말뭉치 NSMC (Naver Sentiment Movie Corpus)
  - 실제 사람이 쓴 리뷰 20만 개 (학습용 15만, 시험용 5만), 긍정/부정 반반
  - 출처: https://github.com/e9t/nsmc  (라이선스 CC0, 자유롭게 사용 가능)
  - 이 코드가 처음 실행될 때 자동으로 내려받는다 (약 20MB)

앞의 연습 예제(가상 데이터 수백 개)와 다른 점
  - 데이터가 실제 리뷰이고 500배 많다 → 처음 보는 표현도 맞힌다
  - 데이터가 많아서 한꺼번에 넣지 않고 256개씩 나눠서 넣는다 (미니배치)
  - 학습용 일부를 검증용으로 떼어 두고, 검증 정확도가 가장 좋을 때의 모델을 고른다 (조기 종료)

설치: pip install torch kiwipiepy pandas scikit-learn
실행: 처음 실행: 내려받기 + 형태소 분석 약 1~2분, 학습은 PC에 따라 에포크당 1~5분
      두 번째부터: 형태소 분석 결과를 저장해 두어서 바로 학습한다
"""

import os
import pickle
import urllib.request
from collections import Counter

import pandas as pd
import torch
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU가 있으면 GPU로
print("사용 장치:", device)

사용 장치: cpu


In [3]:
# 리뷰를 받아올 URL
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/"
# 훈련 데이터 파일명
TRAIN_FILE = "ratings_train.txt"
# 테스트 데이터 파일명
TEST_FILE = "ratings_test.txt"
# 형태소 저장 파일명
TOKEN_CACHE = "nsmc_tokens.pkl"    # 형태소 분석 결과 저장 파일 (2단계 정리 방법을 바꾸면 이 파일을 지우고 다시 실행)
# 모델 저장 파일명
MODEL_FILE = "sentiment_lstm.pt"   # 학습한 모델 저장 파일

# PAD : 형태소 길이가 60이 아닌 문장들의 나머지를 채울 값
# UNK : 새로운 단어
PAD, UNK = 0, 1
MAX_LEN = 60     # 형태소 60개로 길이를 맞춘다 (리뷰의 95%가 60개 이하)
MIN_FREQ = 3     # 학습용 리뷰에 3번 이상 나온 형태소만 사전에 넣는다

In [4]:
# ---------- 1. 데이터 선택 ----------
for name in [TRAIN_FILE, TEST_FILE]:
    if not os.path.exists(name):
        print("내려받는 중:", name)
        urllib.request.urlretrieve(URL + name, name)

# 탭(\t)으로 구분된 파일이다. 열: id, document(리뷰), label(1 긍정, 0 부정)
train_df = pd.read_csv(TRAIN_FILE, sep="\t")
test_df = pd.read_csv(TEST_FILE, sep="\t")
print("학습용:", train_df.shape, "시험용:", test_df.shape)
print(train_df.head())

내려받는 중: ratings_train.txt
내려받는 중: ratings_test.txt
학습용: (150000, 3) 시험용: (50000, 3)
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


In [5]:
# ---------- 2. 사전 준비 ----------
# 2-1. 데이터 정리: 빈 리뷰, 중복 리뷰 빼기
train_df = train_df.dropna(subset=["document"]).drop_duplicates(subset=["document"])
test_df = test_df.dropna(subset=["document"])
print("정리 후 학습용:", len(train_df), "시험용:", len(test_df))

# 2-2. 형태소 분석: 20만 문장이라 시간이 걸려서, 한 번 해 두면 파일로 저장해 둔다
kiwi = Kiwi()


def tokenize(sentence):
    """문장 → 형태소 목록"""
    return [token.form for token in kiwi.tokenize(sentence)]


if os.path.exists(TOKEN_CACHE):
    with open(TOKEN_CACHE, "rb") as f:
        train_tokens, test_tokens = pickle.load(f)
    print("저장해 둔 형태소 분석 결과를 불러왔습니다")
else:
    print("형태소 분석 중... (1~2분)")
    # 문장 목록을 한꺼번에 넘기면 Kiwi가 여러 개를 동시에 처리해서 빠르다
    train_tokens = [[t.form for t in r] for r in kiwi.tokenize(train_df["document"].astype(str).tolist())]
    test_tokens = [[t.form for t in r] for r in kiwi.tokenize(test_df["document"].astype(str).tolist())]
    with open(TOKEN_CACHE, "wb") as f:
        pickle.dump((train_tokens, test_tokens), f)

print("예:", train_df["document"].iloc[0], "→", train_tokens[0])


정리 후 학습용: 146182 시험용: 49997
형태소 분석 중... (1~2분)
예: 아 더빙.. 진짜 짜증나네요 목소리 → ['아', '더빙', '..', '진짜', '짜증', '나', '네요', '목소리']


In [6]:
# ---------- 3. 훈련, 검증, 테스트 분리 ----------
# 시험용(ratings_test)은 이미 따로 있다. 학습용의 10%를 검증용으로 떼어 둔다.
#   학습용: 숫자를 고치는 데 쓴다
#   검증용: 에포크마다 성적을 확인해서, 가장 좋았던 에포크의 모델을 고르는 데 쓴다
#   시험용: 맨 마지막에 딱 한 번 채점한다
y_all = train_df["label"].tolist()
X_train, X_valid, y_train, y_valid = train_test_split(
    train_tokens, y_all, test_size=0.1, random_state=42, stratify=y_all
)
X_test, y_test = test_tokens, test_df["label"].tolist()
print("학습:", len(X_train), "검증:", len(X_valid), "시험:", len(X_test))

학습: 131563 검증: 14619 시험: 49997


In [11]:
# ---------- 4. 학습 ----------
# 4-1. 형태소 사전: 학습용에서만, 자주 나온 형태소만
counter = Counter(t for tokens in X_train for t in tokens)
vocab = {"<PAD>": PAD, "<UNK>": UNK}
for token, count in counter.most_common():
    if count < MIN_FREQ:
        break
    vocab[token] = len(vocab)
print(f"형태소 종류 {len(counter)}개 중 {MIN_FREQ}번 이상 나온 {len(vocab) - 2}개를 사전에 넣었습니다")

# 형태소 사전에 저장된 형태소들 조회(일부)
# count = 0
# for key, value in vocab.items():
#     count += 1
#     print(key, value)
#     if count > 3:
#         break
    

def encode(tokens):
    """형태소 목록 → 길이 MAX_LEN인 번호 목록 (빈칸은 앞에, 너무 길면 앞을 자르고 끝을 남긴다)"""
    ids = [vocab.get(t, UNK) for t in tokens][-MAX_LEN:]
    return [PAD] * (MAX_LEN - len(ids)) + ids


def make_loader(X, y, shuffle):
    """번호 목록과 정답을 묶어서, 256개씩 꺼내 주는 DataLoader를 만든다"""
    data = TensorDataset(torch.tensor([encode(t) for t in X]), torch.tensor(y, dtype=torch.float32))
    return DataLoader(data, batch_size=256, shuffle=shuffle)


train_loader = make_loader(X_train, y_train, shuffle=True)   # 에포크마다 순서를 섞는다
valid_loader = make_loader(X_valid, y_valid, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

형태소 종류 48068개 중 3번 이상 나온 17054개를 사전에 넣었습니다
<PAD> 0
<UNK> 1
이 2
하 3


In [12]:
# 4-2. 모델: 형태소 번호 → 임베딩 → LSTM → 긍정일 점수
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=PAD)   # 형태소 → 숫자 128개
        self.lstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True)
        self.dropout = nn.Dropout(0.3)   # 학습할 때 30%를 무작위로 끈다 (외우기 방지)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)            # (256, 60) → (256, 60, 128)
        _, (h, _) = self.lstm(x)         # 마지막 기억 h: (1, 256, 128)
        return self.fc(self.dropout(h[0])).squeeze(1)   # (256,) 점수. 클수록 긍정


model = SentimentLSTM(len(vocab)).to(device)
print(model)
print("학습할 숫자:", sum(p.numel() for p in model.parameters()))

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def accuracy(loader):
    """DataLoader 전체의 정확도"""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = (torch.sigmoid(model(X)) >= 0.5).float()
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total

SentimentLSTM(
  (embedding): Embedding(17056, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)
학습할 숫자: 2315393


In [13]:
# 4-3. 학습 반복: 한 에포크 = 256개씩 나눠서 학습용 전체를 한 바퀴
EPOCHS = 4
best_valid = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for X, y in train_loader:            # 256개씩 꺼낸다 (미니배치)
        X, y = X.to(device), y.to(device)
        loss = loss_fn(model(X), y)      # 1. 순전파 2. 손실
        optimizer.zero_grad()
        loss.backward()                  # 3. 역전파
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 기울기가 너무 커지지 않게
        optimizer.step()                 # 4. 수정
        total_loss += loss.item() * len(y)

    train_loss = total_loss / len(train_loader.dataset)
    valid_acc = accuracy(valid_loader)
    print(f"에포크 {epoch}: 학습 손실 {train_loss:.4f}, 검증 정확도 {valid_acc:.1%}")

    # 조기 종료: 검증 정확도가 가장 좋았던 모델만 저장해 둔다
    if valid_acc > best_valid:
        best_valid = valid_acc
        torch.save({"model": model.state_dict(), "vocab": vocab, "max_len": MAX_LEN}, MODEL_FILE)
        print("  → 지금까지 가장 좋아서 저장했습니다")

에포크 1: 학습 손실 0.4431, 검증 정확도 83.9%
  → 지금까지 가장 좋아서 저장했습니다
에포크 2: 학습 손실 0.3358, 검증 정확도 85.4%
  → 지금까지 가장 좋아서 저장했습니다
에포크 3: 학습 손실 0.2876, 검증 정확도 86.3%
  → 지금까지 가장 좋아서 저장했습니다
에포크 4: 학습 손실 0.2546, 검증 정확도 86.7%
  → 지금까지 가장 좋아서 저장했습니다


In [14]:
# ---------- 5. 평가 ----------
# 가장 좋았던 모델을 다시 불러와서 시험용으로 딱 한 번 채점한다
saved = torch.load(MODEL_FILE)
model.load_state_dict(saved["model"])
print(f"\n시험 정확도: {accuracy(test_loader):.1%}  (리뷰 약 5만 개, 학습에 한 번도 안 쓴 데이터)")


시험 정확도: 86.2%  (리뷰 약 5만 개, 학습에 한 번도 안 쓴 데이터)


In [17]:
# ---------- 6. 예측 ----------
def predict(sentence):
    """새 문장: 학습 때와 같은 사전 준비(형태소 분석 → 같은 사전 → 같은 길이)를 거친다"""
    ids = torch.tensor([encode(tokenize(sentence))]).to(device)
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(ids)).item()


new_sentences = [
    "현재 마블의 유일한 ‘호프’",
    "톰홀랜드만이 보여줄수있는 스파이더맨의 완성이자 시작. 이제 성장한 스파이더맨 꼭 돌아오길",
    "고맙다 스파이더맨 mcu를 지켜줘서",
    "이것은 절대 1점이 아니다 11점을 주고 싶은 내 마음이다",
    "이 영화는 볼 가치가 없는 영화이다",
    "이 영화를 또 보고 싶네요. 이 영화 안 본 눈 삽니다.",
    "아.. 이 영화를 안 본 눈 삽니다."
]
print()
for s in new_sentences:
    p = predict(s)
    print(f"{'긍정' if p >= 0.5 else '부정'} ({p:5.1%})  {s}")


긍정 (61.7%)  현재 마블의 유일한 ‘호프’
긍정 (90.2%)  톰홀랜드만이 보여줄수있는 스파이더맨의 완성이자 시작. 이제 성장한 스파이더맨 꼭 돌아오길
긍정 (65.8%)  고맙다 스파이더맨 mcu를 지켜줘서
부정 ( 4.1%)  이것은 절대 1점이 아니다 11점을 주고 싶은 내 마음이다
부정 ( 4.0%)  이 영화는 볼 가치가 없는 영화이다
긍정 (93.4%)  이 영화를 또 보고 싶네요. 이 영화 안 본 눈 삽니다.
긍정 (56.0%)  아.. 이 영화를 안 본 눈 삽니다.
